# Step 8 : 数値安定化について

## 1: $Q_{uu,n}$の正則性と正定値化

[Step5 backward pass](/notes/step5.ipynb)の入力修正則の導出において、Q関数の二次近似式の$Q_n$を$\delta u_n$で偏微分して、ゼロとなる停留点を最小点の候補であると導出した。

これが最小点となるための十分条件は、$Q_n$の$\delta u_n$によるHessianが正定値であることであった。

$$
\frac{\partial^2 Q_n}{\partial \ \delta u_n^2} = Q_{uu,n} \succ 0
$$

$Q_{uu,n}$は以下のステージコストの$u$によるHessianと、価値関数の状態$X$によるHessianで構成される。

$$
Q_{uu,n} = \ell_{uu,n} + B_n^{T} V_{XX,n+1} B_n
$$

$\ell_{uu,n} \succ 0, V_{XX,n+1} \succeq 0$ であれば、$Q_{uu,n} \succ 0$ となるが、$V_{XX,n+1},\ell_{uu,n}$は非線形な非凸な問題では負の曲率を持つ場合があり、$Q_{uu,n}$が必ずしも正定値であることは通常保証されない。

そこで次のように、$Q_{uu,n}$が正定値となるように、十分な大きさの正則化係数$\mu \ge 0$を選び、以下のように$\widetilde{Q}_{uu,n}$を構成する。十分な大きさを持つ必要性があるのは、$Q_{uu,n}$が負の曲率を持つ場合、その固有値は負になる。そのため、適当な大きな差の$\mu$でも固有値が負である可能性があるためである。

$$
\widetilde{Q}_{uu,n} = Q_{uu,n} + \mu I
$$

$\widetilde{Q}_{uu,n} \succ 0$ であれば、次の入力修正則$\delta u_n$は、正則化後の局所二次Q関数を$\delta X_n$を固定して、入力の摂動$\delta u_n$について一意に最小化する。

$$
\delta u_n = k_n + K_n \delta X_n
$$

$$
k_n = -\widetilde{Q}_{uu,n}^{-1} Q_{u,n}, \quad K_n = -\widetilde{Q}_{uu,n}^{-1} Q_{uX,n}
$$

### 入力ペナルティ

$Q_n$を$\delta u_n$に関する項でまとめると次のようになる。$C$は$\delta u_n$に依存しない項である。

$$
Q_n(\delta u_n) = (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ Q_{uu,n} \ \delta u_n + C 
$$

これに入力ペナルティを加えると、次のようになり、

$$
\begin{aligned}
\widetilde{Q}_n &= Q_n(\delta u_n) + \frac{\mu}{2} \delta u_n^T \delta u_n \\
&= (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ \left(Q_{uu,n} +  \mu I \right)\ \delta u_n + C 
\end{aligned}
$$

この$\delta u_n$についてのHessianが$\widetilde{Q}_{uu,n}$である。

$$
\widetilde{Q}_{uu,n} = Q_{uu,n} +  \mu I
$$ 

つまり、$\mu$は$\delta u_n$に対する重みであるため、大きくすると、$\delta u_n$を小さくする効果がある。

### 正則化係数$\mu$の調整方法

前節の通り、$\mu$を大きくすると入力修正則が小さくなるので、軌道の改善が遅くなる。そこで、最初から大きな値ではなく、次のように調整を行う。

$\widetilde{Q}_{uu,n}$が正定値行列であるかはCholesky分解の結果より判別することが出来る。

backward passではすでにCholesky分解を用いた$k_n, K_n$の計算と、Cholesky分解を行う$Q_{uu,n}$の対称化処理を行っている。

これに正則化係数$\mu$の調整を加える。

backward passに引数として$\mu$を与える。そしてCholesky分解では対称化した$\widetilde{Q}_{uu,n}(\mu)$の分解を行う。もし分解に失敗したら、$\widetilde{Q}_{uu,n}(\mu)$ は正定値対称行列ではないとなるため、$\mu$を調整してbackward pass を再計算する。

$\mu$の初期値$\mu_{\text{init}} > 0$とし、backward pass を再計算する場合は、$\mu \leftarrow \max(\mu_{\text{init}}, 10 \mu)$ とする。

また、backward pass ではcholesky分解の結果が示すのは$\widetilde{Q}_{uu,n}(\mu)$が正定値行列であることであり、実際にコストが下がるかは forward pass で確認する必要がある。

そこで、forward passのline searchでもコストが下がらない場合、$\mu$を増やしてbackward passを再計算する。コストが下がれば、次の反復では$\mu = \mu_{\text{init}}$として計算を行う。<br>
または前回の反復で用いた$\mu$を用いて、$\mu \leftarrow \max(\mu_{\text{init}}, \mu/10)$のように徐々に減らしながら調整する方法もある。


## 2 : 予測コスト減少量と実際の減少量の比較


### 1時刻での予測コスト変化

局所二次Q関数は次の式である。

$$
\begin{aligned}
Q_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n) \approx & \bar{Q}_n + Q_{X,n}^T \delta X_n + Q_{u,n}^T \delta u_n \\
& + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n + \delta u_n^T Q_{uX,n} \delta X_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
\end{aligned}
$$

この式で、$\delta X_n=0$として入力を$\delta u_n = \alpha k_n$だけ変化させると、以下となる。

$$
Q_n(\bar{X}_n, \bar{u}_n+ \alpha k_n) - Q_n(\bar{X}_n, \bar{u}_n) \approx \alpha Q_{u,n}^T  k_n + \frac{1}{2} \alpha^2 k_n^T Q_{uu,n} k_n
$$ 

上式の右辺はコストの変化量であるため、負なら減少を意味する。

以下のように$d_{1,n}, d_{2,n}$を定義すれば、

$$
d_{1,n} =Q_{u,n}^T  k_n , \quad d_{2,n} = \frac{1}{2} k_n^T Q_{uu,n} k_n
$$

次のようにコストの変化分を記述できる。

$$
\Delta Q_n(\alpha) = \alpha d_{1,n} + \alpha^2 d_{2,n}
$$

正則化を行っていない$Q_{uu,n}$では、$Q_{uu,n} k_n = - Q_{u,n}$であるため、コストの変化分は以下となる。

$$
\begin{aligned}
\Delta Q_n(\alpha) &= - \alpha (Q_{uu,n} k_n)^T  k_n + \frac{1}{2} \alpha^2 k_n^T Q_{uu,n} k_n \\
&= - \alpha k_n^T Q_{uu,n}^T  k_n + \frac{1}{2} \alpha^2 k_n^T Q_{uu,n}  k_n \\
&= - \alpha (1 - \frac{\alpha}{2}) k_n^T Q_{uu,n} k_n 
\end{aligned}
$$

$Q_{uu,n} \succ 0, k_n \neq 0, 0 < \alpha \le 1$ の条件では$\Delta Q_n(\alpha)< 0$となる。 




### 2.1 ホライゾン全体の予測コスト減少量

#### 2.2 $Q_{uu,n}$の正則化なしの場合

以下では、次の条件で予測コストの減少量を考える。

- 正則化なしの$Q_{uu,n} \succ 0$ 
- 初期状態の摂動 $\delta X_0 = 0$
- forward passでは線形化モデルで生成した摂動軌道を計算
- 予測コストは総コストの近似式で計算

Q関数の二次近似式$\widehat{Q}_n$を以下で定義する。

$$
\begin{aligned}
\widehat{Q}_n(\delta X_n, \delta u_n) \coloneqq & \bar{Q}_n + Q_{X,n}^T \delta X_n + Q_{u,n}^T \delta u_n \\
& + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n + \delta u_n^T Q_{uX,n} \delta X_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
\end{aligned}
$$

上式は以下の式を用いて、

$$
C_n(\delta X_n) = \bar{Q}_n + Q_{X,n}^T \delta X_n+ \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n , \quad b_n = Q_{u,n} + Q_{uX,n} \delta X_n
$$

以下のように書ける。

$$
\widehat{Q}_n(\delta X_n, \delta u_n) = C_n(\delta X_n) + b_n^T \delta u_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
$$

$\delta X_n$を固定し、$\widehat{Q}_n$を最小化する入力を $\delta u_n^\star$ と書くと以下である。

$$
\delta u_n^\star = -Q_{uu,n}^{-1} b_n = k_n + K_n \delta X_n
$$

よって、以下となる。

$$
b_n = - Q_{uu,n} \delta u_n^\star
$$

これより、次のようになる。

$$
\begin{aligned}
\widehat{Q}_n(\delta X_n, \delta u_n) &= C_n(\delta X_n) - (\delta u_n^\star)^T Q_{uu,n} \delta u_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n \\
&= C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T Q_{uu,n} \delta u_n^\star \\
& \quad + \frac{1}{2} (\delta u_n - \delta u_n^\star) ^T Q_{uu,n} (\delta u_n - \delta u_n^\star) \\
\end{aligned}
$$

価値関数の近似式は、$\widehat{Q}_n$を$\delta u_n$について最小化したものである。

$$
\widehat{V}_n(\delta X_n) = \underset{\delta u_n}{\min} \widehat{Q}_n(\delta X_n, \delta u_n)
$$

よって、上式の$\delta u_n$に$\delta u_n^\star$を代入したものが$\widehat{V}_n(\delta X_n)$である。

$$
\begin{aligned}
\widehat{V}_n(\delta X_n) &= \widehat{Q}_n(\delta X_n, \delta u_n^\star) \\
&= C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T Q_{uu,n} \delta u_n^\star + \frac{1}{2} (\delta u_n^\star - \delta u_n^\star) ^T Q_{uu,n} (\delta u_n^\star - \delta u_n^\star) \\
&= C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T Q_{uu,n} \delta u_n^\star  
\end{aligned}
$$

これより、$\widehat{Q}_n(\delta X_n, \delta u_n)$は価値関数を用いて次のように記述できる。

$$
\begin{aligned}
\widehat{Q}_n(\delta X_n, \delta u_n) =\widehat{V}_n(\delta X_n) + \frac{1}{2} (\delta u_n - \delta u_n^\star) ^T Q_{uu,n} (\delta u_n - \delta u_n^\star) 
\end{aligned}
$$

ここに、forward passの修正項を代入すると、

$$
\delta u_n = \alpha k_n + K_n \delta X_n
$$

$$
\begin{aligned}
\widehat{Q}_n(\delta X_n, \delta u_n) & = \widehat{V}_n(\delta X_n) \\
& + \frac{1}{2} (\alpha k_n + \cancel{K_n \delta X_n} -  k_n - \cancel{K_n \delta X_n}) ^T Q_{uu,n} (\alpha k_n + \cancel{K_n \delta X_n} -  k_n - \cancel{K_n \delta X_n}) \\
&= \widehat{V}_n(\delta X_n) + \frac{1}{2}(- (1- \alpha) k_n)^T Q_{uu,n} (- (1- \alpha) k_n) \\
&= \widehat{V}_n(\delta X_n) + \frac{1}{2} (1 - \alpha)^2 k_n^T Q_{uu,n} k_n \\
\end{aligned}
$$

元の非線形なQ関数は次の定義である。

$$
\begin{aligned}
Q_n(X_n, u_n) = \ell_n(X_n, u_n) + V_{n+1}(X_{n+1}) , \quad X_{n+1} &= f(X_n, u_n)
\end{aligned}
$$


これを局所近似したQ関数は以下となる。$\widehat{\ell}_n$はステージコストの二次近似である。

$$
\widehat{Q}_n(\delta X_n , \delta u_n) = \widehat{\ell}_n (\delta X_n, \delta u_n) + \widehat{V}_{n+1}(\delta X_{n+1}) , \quad \delta X_{n+1} = A_n \delta X_n + B_n \delta u_n
$$

上式は先ほどまとめた$\widehat{Q}_n(\delta X_n , \delta u_n)$と同じであるため、ステージコストの二次近似は次のようになる。

$$
\begin{array}{c}
\widehat{V}_n(\delta X_n) + \frac{1}{2} (1 - \alpha)^2 k_n^T Q_{uu,n} k_n = \widehat{\ell}_n (\delta X_n, \delta u_n) + \widehat{V}_{n+1}(\delta X_{n+1}) \\
\downarrow \\
\widehat{\ell}_n (\delta X_n, \delta u_n) = \widehat{V}_n(\delta X_n) - \widehat{V}_{n+1}(\delta X_{n+1})+ \frac{1}{2} (1 - \alpha)^2 k_n^T Q_{uu,n} k_n 
\end{array}
$$

予測コストは、総コストの近似式を用いて次のように計算する。

$$
\widehat{J}(\alpha) = \widehat{\phi}(\delta X_N) + \sum_{n=0}^{N-1} \widehat{\ell}_n
$$

$\widehat{\ell}_n$ を$n=0, ... , N-1$で加算していく。価値関数の部分だけを見ると、

$$
\begin{aligned}
 &[\widehat{V}_0(\delta X_0) - \widehat{V}_1(\delta X_1)]\\
+&[\widehat{V}_1(\delta X_1) - \widehat{V}_2(\delta X_2)]\\
+&\cdots\\
+&[\widehat{V}_{N-1}(\delta X_{N-1}) - \widehat{V}_N(\delta X_N)]\\
\end{aligned}
$$

であるため、途中の項が打ち消しあい、次のようになる。

$$
\sum_{n=0}^{N-1} \widehat{\ell}_n = \widehat{V}_0(\delta X_0) - \widehat{V}_N(\delta X_N) + \frac{1}{2} (1 - \alpha)^2 \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n
$$ 

$\widehat{V}_N(\delta X_N) = \widehat{\phi}(\delta X_N)$ であるため、予測コストは以下となる。

$$
\widehat{J}(\alpha) = \widehat{V}_0(\delta X_0)  + \frac{1}{2} (1 - \alpha)^2 \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n
$$

初期摂動の状態は$\delta X_0 = 0$とし、線形化モデルでforward passを計算するので、$\alpha=0$とすると、以下のように状態摂動、入力摂動ともにゼロとなる。

$$
\begin{aligned}
 (\delta X_0 = X_0^{\text{new}} - \bar{X}_0 = 0) &\rightarrow (\delta u_0 = K_0 \delta X_0 = 0) \\
\rightarrow  (\delta X_1 = A_0 \delta X_0 + B_0 \delta u_0 = 0 ) &\rightarrow (\delta u_1 = K_1 \delta X_1 = 0) \\
\vdots \\
(\delta X_{n+1} = A_n \delta X_n + B_n \delta u_n = 0 ) &\rightarrow (\delta u_{n+1} = K_{n+1} \delta X_{n+1} = 0)
\end{aligned}
$$

よって、基準軌道による予測コスト$\widehat{J}(0)$となる。

$$
\widehat{J}(0) = \widehat{V}_0(\delta X_0)  + \frac{1}{2} \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n
$$

よって、$\alpha k_n$ による、予測コスト変化$\Delta \widehat{J}(\alpha)$は$\widehat{J}(\alpha) - \widehat{J}(0)$となるため、

$$
\Delta \widehat{J}(\alpha) = - \alpha (1 - \frac{\alpha}{2}) \sum_{n=0}^{N-1} k_n^T Q_{uu,n} k_n 
$$

これを予測減少量$D_{\text{pred}}(\alpha)$として以下で定義する。

$$
D_{\text{pred}}(\alpha) = - \Delta \widehat{J}(\alpha)
$$


#### 実際のコスト減少量との比較

基準軌道と候補軌道の総コストを、元の非線形コスト関数で計算を行う。候補軌道は非線形モデルによるforward passで求める。

$$
J_{\text{old}} = \phi(\bar{X}_N) + \sum_{n=0}^{N-1} \ell_n(\bar{X}_n, \bar{u}_n)
$$

$$
J_{\text{new}}(\alpha) = \phi({X}_N^{\text{new}}) + \sum_{n=0}^{N-1} \ell_n({X}_n^{\text{new}}, {u}_n^{\text{new}})
$$

実際のコスト減少量$D_{\text{actual}}(\alpha)$は以下となる。

$$
D_{\text{actual}}(\alpha) = J_{\text{old}} - J_{\text{new}}(\alpha)
$$

よって、$D_{\text{pred}}(\alpha) >0$であるとき、予測減少量に対する実際のコスト減少率$\rho(\alpha)$は以下で計算できる。この計算は同じ$\alpha$で計算を行う。$D_{\text{pred}}(\alpha)$がゼロまたは、非常に小さい場合は$\rho(\alpha)$は計算を行わない。

$$
\rho(\alpha) = \frac{D_{\text{actual}}(\alpha)}{D_{\text{pred}}(\alpha)}
$$

#### $\rho(\alpha)$を用いた調整

$\rho$は局所近似モデルによる予測コスト減少量に対する、非線形モデルで評価した実際のコスト減少量の比である。

よって、次のような意味になる。

|$\rho$|意味|
|:---:|:---|
|1に近い| 今回の候補では、予測に近いコスト減少が得られた|
|正だが小さい| コストは減ったが、予測ほど改善しなかった|
|ゼロ以下| コストが減らなかった。負の場合は増加した|
|1より大きい| 予測以上にコストが減った。|

この$\rho$は次のような判断を行う。

1. ラインサーチで候補を採用するかを決める  
    各$\alpha$でforward pass を行い、
        $$
            \rho(\alpha) > \eta, 0 < \eta < 1
        $$
    であればその候補軌道を次の基準軌道として採用する。例えば $\eta = 0.1$ とすると、予測減少量の10%を超える実際の減少が得られたら採用するということになる。<br>
    満たさなければ、より小さな$\alpha$を試す。より小さな$\alpha$とする意味は、候補軌道をより基準軌道へ近づけることになる。ただこの処理自体は従来のラインサーチと同じだが、採用条件をより明確にするということである。
2. 正則化ありの場合、次の反復の$\mu$を調整する
    - $\rho$が十分に大きい: 予測に対して十分な改善が得られたため、$\mu$を減らし、入力修正のへ抑制を弱める。次の反復では、より大きな修正を試せるようにする。
    - $\rho$が小さい：予測ほど改善していないので、$\mu$を増やし、入力修正を抑える。基準軌道の近傍で、より慎重に改善を試す。

これらは調整方針の一例であり、$\mu$を変更しても、次の候補のコスト減少や予測精度が保証されるわけではない。    

#### $Q_{uu,n}$を正則化する場合

以下では、次の条件で予測コストの減少量を考える。
これ以外は正則化なしと同じ条件である。

- 正則化ありの$\widetilde{Q}_{uu,n} \succ 0$ 


Q関数の二次近似式$\widehat{Q}_n$に入力ペナルティを加えたものを$\widetilde{\widehat{Q}}_n$として定義する。この節の$\widehat{Q}_n$は、後述する正則化後の次時刻の価値関数を用いた$\widehat{\ell}_n + \widetilde{\widehat{V}}_{n+1}$を表す。

$$
\begin{aligned}
\widetilde{\widehat{Q}}_n(\delta X_n, \delta u_n) & \coloneqq  {\widehat{Q}}_n(\delta X_n, \delta u_n) + \frac{\mu}{2}\delta u_n^T \delta u_n \\
&= C_n(\delta X_n) + b_n^T \delta u_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n + \frac{\mu}{2}\delta u_n^T \delta u_n \\
&= C_n(\delta X_n) + b_n^T \delta u_n + \frac{1}{2} \delta u_n^T (Q_{uu,n} + \mu \boldsymbol{I}) \delta u_n \\
&= C_n(\delta X_n) + b_n^T \delta u_n + \frac{1}{2} \delta u_n^T \widetilde{Q}_{uu,n} \delta u_n \\
\end{aligned}
$$

backward pass で計算される、 $\delta X_n$ を固定した、$\widetilde{\widehat{Q}}_n$を最小化する入力は$\widetilde{Q}_{uu,n}$を用いて次の $\delta u_n^\star$である。

$$
\begin{aligned}
\delta u_n^\star &= -\widetilde{Q}_{uu,n}^{-1} Q_{u,n} - \widetilde{Q}_{uu,n}^{-1} Q_{uX,n} \delta X_n \\
&= k_n  +K_n \delta X_n \\
\end{aligned}
$$

$b_n = Q_{u,n} + Q_{uX,n} \delta X_n$より、

$$
\delta u_n^\star = -\widetilde{Q}_{uu,n}^{-1} b_n \Rightarrow b_n = - \widetilde{Q}_{uu,n} \delta u_n^\star
$$

これより、

$$
\begin{aligned}
\widetilde{\widehat{Q}}_n(\delta X_n, \delta u_n) &= C_n(\delta X_n) - (\delta u_n^\star)^T \widetilde{Q}_{uu,n} \delta u_n + \frac{1}{2} \delta u_n^T \widetilde{Q}_{uu,n} \delta u_n \\
&= C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T  \widetilde{Q}_{uu,n} \delta u_n^\star \\
& \quad + \frac{1}{2} (\delta u_n - \delta u_n^\star) ^T \widetilde{Q}_{uu,n} (\delta u_n - \delta u_n^\star) \\
\end{aligned}
$$

正則化を行う場合の価値関数の近似式は次となる。

$$
\widetilde{\widehat{V}}_n(\delta X_n) = \underset{\delta u_n}{\min} \left( \widehat{Q}_n(\delta X_n, \delta u_n) + \frac{\mu}{2} \delta u_n^T \delta u_n \right) = \underset{\delta u_n}{\min} \widetilde{\widehat{Q}}_n(\delta X_n, \delta u_n)
$$

よって上式の$\widetilde{\widehat{Q}}$の$\delta u_n$に$\delta u_n^\star$を代入したものが、正則化を行うときの価値関数の近似式である。

$$
\begin{aligned}
\widetilde{\widehat{V}}_n(\delta X_n) &= \widetilde{\widehat{Q}}_n(\delta X_n, \delta u_n^\star) \\
&=C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T  \widetilde{Q}_{uu,n} \delta u_n^\star + \frac{1}{2} (\delta u_n^\star - \delta u_n^\star) ^T \widetilde{Q}_{uu,n} (\delta u_n^\star - \delta u_n^\star) \\
&= C_n(\delta X_n) - \frac{1}{2} (\delta u_n^\star)^T  \widetilde{Q}_{uu,n} \delta u_n^\star
\end{aligned}
$$

この価値関数の近似式を用いて平方完成で構成した$\widetilde{\widehat{Q}}_n$は次のように記述できる。

$$
\begin{aligned}
\widetilde{\widehat{Q}}_n(\delta X_n, \delta u_n) = \widetilde{\widehat{V}}_n(\delta X_n) + \frac{1}{2} (\delta u_n - \delta u_n^\star) ^T \widetilde{Q}_{uu,n} (\delta u_n - \delta u_n^\star) \\
\end{aligned}
$$

これに forward passの修正項$\delta u_n$を代入すると、

$$
\delta u_n = \alpha k_n + K_n \delta X_n
$$

より、以下となる。

$$
\begin{aligned}
\widetilde{\widehat{Q}}_n(\delta X_n, \delta u_n) = \widetilde{\widehat{V}}_n(\delta X_n) + \frac{1}{2} (1- \alpha)^2 k_n^T \widetilde{Q}_{uu,n} k_n \\
\end{aligned}
$$

前節の正則化なしの局所近似したQ関数は以下であった。

$$
\widehat{Q}_n(\delta X_n , \delta u_n) = \widehat{\ell}_n (\delta X_n, \delta u_n) + \widehat{V}_{n+1}(\delta X_{n+1}) , \quad \delta X_{n+1} = A_n \delta X_n + B_n \delta u_n
$$

正則化ありでは、現在時刻$n$の入力ペナルティに加え、時刻$n+1$以降の入力ペナルティを含む価値関数$\widetilde{\widehat{V}}_{n+1}$を用いる。

$$
\widetilde{\widehat{Q}}_n(\delta X_n , \delta u_n) = \widetilde{\widehat{\ell}}_n(\delta X_n, \delta u_n) + \widetilde{\widehat{V}}_{n+1}(\delta X_{n+1}) , \quad \delta X_{n+1} = A_n \delta X_n + B_n \delta u_n
$$

$$
\widetilde{\widehat{\ell}}_n(\delta X_n, \delta u_n) =  \widehat{\ell}_n (\delta X_n, \delta u_n) + \frac{\mu}{2} \delta u_n^T \delta u_n 
$$

平方完成を用いた$\widetilde{\widehat{Q}}_n$とQ関数の局所近似の定義$\widetilde{\widehat{Q}}_n$より、予想コストで積算するステージコストの二次近似は以下のようになる。

$$
\widetilde{\widehat{\ell}}_n(\delta X_n, \delta u_n) = \widetilde{\widehat{V}}_n(\delta X_n) - \widetilde{\widehat{V}}_{n+1}(\delta X_{n+1}) + \frac{1}{2} (1- \alpha)^2 k_n^T \widetilde{Q}_{uu,n} k_n
$$

正則化ありの予想コストは、総コストの近似式を用いて次のように計算する。

$$
\widetilde{\widehat{J}}(\alpha, \mu) = \widehat{\phi}(\delta X_N) + \sum_{n=0}^{N-1} \widetilde{\widehat{\ell}}_n
$$

$\widetilde{\widehat{\ell}}_n(\delta X_n, \delta u_n)$ を $n = 0,...,N-1$ で加算していくと、価値関数は途中の項が打ち消しあうので、次のようになる。

$$
\sum_{n=0}^{N-1} \widetilde{\widehat{\ell}}_n(\delta X_n, \delta u_n) = \widetilde{\widehat{V}}_0(\delta X_0) - \widetilde{\widehat{V}}_N(\delta X_N) +  \frac{1}{2} (1- \alpha)^2 \sum_{n=0}^{N-1} k_n^T \widetilde{Q}_{uu,n} k_n
$$

$\widetilde{\widehat{V}}_N(\delta X_N) = \widehat{\phi}(\delta X_N)$ であるため、予想コストは以下となる。この予想コストは入力ペナルティが考慮された候補軌道で計算したコストである。

$$
\begin{aligned}
\widetilde{\widehat{J}}(\alpha, \mu) &= \widehat{\phi}(\delta X_N) + \sum_{n=0}^{N-1} \widetilde{\widehat{\ell}}_n (\delta X_n, \delta u_n) \\
&= \widetilde{\widehat{V}}_0(\delta X_0) + \frac{1}{2} (1- \alpha)^2 \sum_{n=0}^{N-1} k_n^T \widetilde{Q}_{uu,n} k_n
\end{aligned}
$$

$\widetilde{\widehat{V}}_0(\delta X_0)$は$\delta X_0 = 0$であるため、以下の式である。

$$
\begin{aligned}
\widetilde{\widehat{V}}_0(\delta X_0) &= C_0(\delta X_0) - \frac{1}{2} (\delta u_0^\star)^T  \widetilde{Q}_{uu,0} \delta u_0^\star \\
&= \bar{Q}_0 + Q_{X,0}^T \delta X_0+  \frac{1}{2} \delta X_0^T Q_{XX,0} \delta X_0 - \frac{1}{2} (\delta u_0^\star)^T  \widetilde{Q}_{uu,0} \delta u_0^\star \\
&= \bar{Q}_0 - \frac{1}{2} (k_0 + K_0 \delta X_0)^T  \widetilde{Q}_{uu,0} (k_0 + K_0 \delta X_0) \\
&= \bar{Q}_0 - \frac{1}{2} k_0^T  \widetilde{Q}_{uu,0} k_0  \\
\end{aligned}
$$

よって、

$$
\widetilde{\widehat{J}}(\alpha, \mu) = \bar{Q}_0 - \frac{1}{2} k_0^T  \widetilde{Q}_{uu,0} k_0 + \frac{1}{2} (1- \alpha)^2 \sum_{n=0}^{N-1} k_n^T \widetilde{Q}_{uu,n} k_n
$$

$\alpha=0$では以下の式となる。

$$
\widetilde{\widehat{J}}(0, \mu) = \bar{Q}_0  - \frac{1}{2} k_0^T  \widetilde{Q}_{uu,0} k_0 + \frac{1}{2} \sum_{n=0}^{N-1} k_n^T \widetilde{Q}_{uu,n} k_n
$$

上式の$k_n, \widetilde{Q}_{uu,n}$は$\mu$に依存するが、$\delta X_0=0$で$\alpha=0$の場合、正則化なしの節で示したように、状態摂動と入力摂動はゼロになる。よって、以下のように$\mu$に依存しない、基準軌道で計算した予想コスト$\widehat{J}_{\text{old}}$となる。

$$
\begin{aligned}
\widetilde{\widehat{J}}(0, \mu) &= \widehat{\phi}(\delta X_N) + \sum_{n=0}^{N-1} \widetilde{\widehat{\ell}}_n (\delta X_n, \delta u_n)\\
&= \widehat{\phi}(\delta X_N) + \sum_{n=0}^{N-1} \left[ \widehat{\ell}_n (\delta X_n, \delta u_n) + \frac{\mu}{2} \delta u_n^T \delta u_n \right] \\
&= \widehat{\phi}(0) + \sum_{n=0}^{N-1} \left[ \widehat{\ell}_n (0, 0) + \frac{\mu}{2} 0^T 0 \right] \\
&= \widehat{J}_{\text{old}} = \widetilde{\widehat{J}}(0)
\end{aligned}
$$


これより、$\alpha k_n$ による、予測減少量は以下として定義できる。

$$
\widetilde{D}_{\text{pred}}(\alpha, \mu) = -(\widetilde{\widehat{J}}(\alpha,\mu) - \widetilde{\widehat{J}}(0))
= \alpha (1- \frac{\alpha}{2}) \sum_{n=0}^{N-1} k_n^T \widetilde{Q}_{uu,n} k_n 
$$


#### 実際のコスト減少量との比較

基準軌道と候補軌道の総コストを、元の非線形コスト関数で計算を行う。候補軌道は非線形モデルによるforward passで求める。ここで求める$\widetilde{J}_{\text{new}}$は$Q_{uu,n}$を正則化した入力修正則で計算された軌道によるコストである。

$$
J_{\text{old}} = \phi(\bar{X}_N) + \sum_{n=0}^{N-1} \ell_n(\bar{X}_n, \bar{u}_n)
$$

$$
\widetilde{J}_{\text{new}}(\alpha) = \phi({X}_N^{\text{new}}) + \sum_{n=0}^{N-1} \ell_n({X}_n^{\text{new}}, {u}_n^{\text{new}})
$$

実際のコスト減少量$D_{\text{actual}}(\alpha)$は以下となる。

$$
\widetilde{D}_{\text{actual}}(\alpha) = J_{\text{old}} - \widetilde{J}_{\text{new}}(\alpha)
$$

よって、$\widetilde{D}_{\text{pred}}(\alpha,\mu) >0$であるとき、予測減少量に対する実際のコスト減少率$\rho(\alpha)$は以下で計算できる。この計算は同じ$\alpha$で計算を行う。$\widetilde{D}_{\text{pred}}(\alpha)$がゼロまたは、非常に小さい場合は$\rho(\alpha)$は計算を行わない。

$$
\widetilde{\rho}(\alpha,\mu) = \frac{\widetilde{D}_{\text{actual}}(\alpha)}{\widetilde{D}_{\text{pred}}(\alpha,\mu)}
$$